# Chapter 18. Prediction validation and model scope

**Learning objective.** Match the validation partition to the intended new unit, keep preprocessing inside training folds, report calibration and domain errors, and separate predictive evidence from explanation.

**Estimated time:** 120 minutes. **Exercise:** 18.4. Row-random, speaker-, item-, session-, and corpus-held-out estimates answer different deployment questions.

## Input and output contract

Inputs are a complete synthetic token-level feature table, bounded model registry, fold policy, deployment domain, preprocessing recipe, outcome schema, claim map, and the first notebook's diagnostics. Outputs include auditable fold assignments, leakage checks, fold-level performance and calibration, domain errors, model comparison, four-dimensional scope gate, and provenance.

In [ ]:
find_repository<-function(start=getwd()){current<-normalizePath(start,mustWork=TRUE);repeat{if(file.exists(file.path(current, "companion", "data", "MANIFEST.tsv")))return(current);parent<-dirname(current);if(identical(parent,current))stop("Run from the repository root or a subdirectory.");current<-parent}}
REPOSITORY_ROOT<-find_repository();DATA_ROOT<-file.path(REPOSITORY_ROOT,"companion","data","ch18","synthetic_complex");if(!dir.exists(DATA_ROOT))stop("Chapter 18 fixture absent. Run companion/scripts/generate_ch18_fixture.py explicitly; no data are substituted silently.");source(file.path(REPOSITORY_ROOT,"companion","r","ch18.R"));cat("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — complex-model workflow test, not population evidence.
")

In [ ]:
DYNAMIC<-file.path(REPOSITORY_ROOT,"companion","outputs","ch18","01_dynamic_error_and_missingness");if(!dir.exists(DYNAMIC))stop("Run the first Chapter 18 notebook first.")
OUTPUT<-file.path(REPOSITORY_ROOT,"companion","outputs","ch18","02_prediction_validation_and_scope")
result<-run_prediction_validation_scope(file.path(DATA_ROOT,"prediction_features.tsv"),file.path(DATA_ROOT,"candidate_models.tsv"),file.path(DATA_ROOT,"fold_policy.yaml"),file.path(DATA_ROOT,"deployment_domain.yaml"),file.path(DATA_ROOT,"preprocessing_recipe.yaml"),file.path(DATA_ROOT,"outcome_schema.yaml"),file.path(DATA_ROOT,"claim_map.yaml"),DYNAMIC,OUTPUT)
stopifnot(nrow(result$folds)==720,result$gate$decision=="restrict",!result$gate$row_random_is_new_speaker_evidence,!result$gate$predictive_success_implies_causation)
print(result$comparison);print(result$audit);print(result$gate)

## Learner decision

Compare the five validation schemes without treating them as interchangeable accuracy estimates. Identify every unit that overlaps between train and test, then state the deployment claim supported by each split. Write the final scope in four parts—population, measurement, predictor, and goal—and explain why predictive success does not identify a phonetic mechanism.

In [ ]:
required<-c("fold_assignments.tsv","leakage_audit.tsv","validation_performance.tsv","calibration.tsv","domain_errors.tsv","model_comparison.tsv","model_scope_gate.yaml","run_provenance.json");stopifnot(all(file.exists(file.path(OUTPUT,required))),result$decisions$all_preprocessing_inside_folds);print(result$decisions)